# Пример 2 из 5: корзина задач (evaluation suite) — тема 13

Корзина задач — **центральный артефакт** eval: именно она определяет, что считается качеством. Каждая задача — это не только текст запроса, а **полностью описанный сценарий**: запрос, начальное состояние среды, применимые политики и **ожидаемое итоговое состояние**.

Собираем корзину в два шага: **вручную** (мы знаем, что агент должен уметь) и **генерацией БЯМ** по матрице «функция × сценарий × персона» с обязательной ручной проверкой. Готовую корзину версионируем и сохраняем.

In [1]:
import os, json
from pathlib import Path
import pandas as pd
from pydantic import BaseModel, ValidationError
from openai import OpenAI
import eval_env as E

client = OpenAI(api_key=os.environ.get("LLM_API_KEY","") or "EMPTY",
                base_url=os.environ.get("LLM_BASE_URL","http://localhost:8000/v1"))
MODEL = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")
print("Готово.")

Готово.


## Структура задачи

Схема Pydantic фиксирует полную структуру. Поля для грейдинга (ожидаемое состояние, ожидаемые инструменты, политика) заранее закладываются в задачу — всё, что помогает объективной проверке.

In [2]:
class Task(BaseModel):
    id: str
    query: str                       # что вводит пользователь
    category: str                    # функция: order_info | cancel | change | policy | refuse
    persona: str                     # новичок | раздражённый | нечёткий | обычный
    difficulty: str                  # easy | medium | hard
    needs_dialog: bool = False       # многоходовый ли сценарий (нужен Iron User)
    policy: str | None = None        # применимая политика (cancellation/change/return)
    expected_state: dict = {}        # ожидаемое итоговое состояние среды (пусто = без изменений)
    expected_tools: list[str] = []   # какие инструменты агент ДОЛЖЕН вызвать
    reference: str = ""              # эталон: почему решаемо / корректный исход

print("Схема задачи определена.")

Схема задачи определена.


## Шаг 1. Ручная корзина

Восемь задач, покрывающих функции и уровни сложности, включая **двунаправленные** (где корректный исход — отказ: отмена отправленного заказа) и **граничные** случаи.

In [3]:
manual = [
    Task(id="t01", query="Покажи информацию по заказу ORD-1001", category="order_info",
         persona="обычный", difficulty="easy", expected_tools=["get_order"],
         reference="read-only, состояние не меняется"),
    Task(id="t02", query="Есть ли USB-C хаб в наличии?", category="order_info",
         persona="новичок", difficulty="easy", expected_tools=["search_products"],
         reference="поиск товара, без изменений"),
    Task(id="t03", query="Отмени заказ ORD-1003", category="cancel", persona="обычный",
         difficulty="medium", policy="cancellation", expected_tools=["get_policy", "cancel_order"],
         expected_state={"ORD-1003": {"status": "cancelled"}},
         reference="processing -> отмена разрешена"),
    Task(id="t04", query="Отмени заказ ORD-1002", category="refuse", persona="раздражённый",
         difficulty="medium", policy="cancellation", expected_tools=["get_order"],
         expected_state={},   # ДВУНАПРАВЛЕННАЯ: отправленный заказ отменять НЕЛЬЗЯ
         reference="shipped -> агент обязан отказать, состояние не меняется"),
    Task(id="t05", query="Измени количество в заказе ORD-1003 на 3 штуки", category="change",
         persona="обычный", difficulty="medium", policy="change",
         expected_tools=["get_policy", "change_order_qty"],
         expected_state={"ORD-1003": {"qty": 3}},
         reference="processing -> изменение разрешено"),
    Task(id="t06", query="Увеличь количество в доставленном заказе ORD-1001 до 2", category="refuse",
         persona="нечёткий", difficulty="hard", policy="change", expected_tools=["get_order"],
         expected_state={},   # ДВУНАПРАВЛЕННАЯ: delivered изменять нельзя
         reference="delivered -> изменение запрещено, отказ"),
    Task(id="t07", query="Какие правила возврата товара?", category="policy", persona="новичок",
         difficulty="easy", policy="return", expected_tools=["get_policy"],
         reference="справка по политике"),
    Task(id="t08", query="Отмени мой последний заказ", category="cancel", persona="нечёткий",
         difficulty="hard", needs_dialog=True, policy="cancellation",
         expected_tools=["get_order"], reference="неоднозначно -> нужен уточняющий вопрос (диалог)"),
]
print("Ручных задач:", len(manual))
pd.Series([t.difficulty for t in manual]).value_counts().to_frame("count")

Ручных задач: 8


,count
easy,3
medium,3
hard,2


## Шаг 2. Генерация БЯМ по матрице «функция × сценарий × персона»

Просим модель сгенерировать ещё задачи в том же формате JSON. Каждую сгенерированную задачу **валидируем схемой** и оставляем только корректные (аналог ручной проверки: решаема ли, в домене ли).

In [4]:
GEN_PROMPT = f"""Ты помогаешь составить корзину тестовых задач для агента поддержки интернет-магазина.
Среда: заказы ORD-1001 (delivered), ORD-1002 (shipped), ORD-1003 (processing); товары P-100, P-300, P-400.
Политики: отмена/изменение — только до отправки; возврат — 14 дней после доставки.
Инструменты: get_order, search_products, get_policy, cancel_order, change_order_qty.

Сгенерируй 8 РАЗНЫХ задач по матрице «функция(order_info/cancel/change/policy/refuse) x
сценарий(норма/неоднозначность/граница) x персона(новичок/раздражённый/нечёткий/обычный)».
Включи негативные (refuse) случаи. Верни ТОЛЬКО JSON-массив объектов с полями:
id, query, category, persona, difficulty(easy/medium/hard), needs_dialog(bool),
policy(cancellation/change/return или null), expected_state(dict), expected_tools(list), reference.
id начинай с "g"."""

raw = client.chat.completions.create(model=MODEL, max_tokens=1500,
    messages=[{"role":"user","content":GEN_PROMPT}]).choices[0].message.content
arr = json.loads(raw[raw.find("["): raw.rfind("]")+1])

generated, rejected = [], 0
for obj in arr:
    try:
        generated.append(Task(**obj))     # валидация схемой = ручная проверка формата
    except (ValidationError, TypeError):
        rejected += 1
print(f"Сгенерировано валидных: {len(generated)}, отклонено: {rejected}")
for t in generated[:3]:
    print(f"  [{t.difficulty}/{t.category}] {t.query[:60]}")

Сгенерировано валидных: 8, отклонено: 0
  [easy/cancel] Здравствуйте, я новичок. Мой заказ ORD-1003 в обработке. Мож
  [medium/change] Привет, я раздражённый. Хочу изменить количество товара в за
  [easy/policy] Я обычный покупатель. Могу ли я вернуть товар из заказа ORD-


## Шаг 3. Объединение, баланс и версионирование

Объединяем ручные и сгенерированные задачи, снимаем дубли id, смотрим **баланс классов** (по сложности и категории) и сохраняем корзину как **версионированный** артефакт `data/eval_suite.json`.

In [5]:
suite = {t.id: t for t in manual}
for t in generated:
    suite.setdefault(t.id, t)
tasks = list(suite.values())
print("Всего задач в корзине:", len(tasks))

df = pd.DataFrame([{"difficulty": t.difficulty, "category": t.category, "persona": t.persona}
                   for t in tasks])
print("\nБаланс по сложности:"); print(df["difficulty"].value_counts().to_string())
print("\nБаланс по категории:"); print(df["category"].value_counts().to_string())

Path("data").mkdir(exist_ok=True)
payload = {"version": "v1", "n_tasks": len(tasks),
           "tasks": [t.model_dump() for t in tasks]}
Path("data/eval_suite.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2))
print("\nСохранено: data/eval_suite.json (версия v1,", len(tasks), "задач)")

Всего задач в корзине: 16

Баланс по сложности:
difficulty
easy      7
medium    6
hard      3

Баланс по категории:
category
refuse        5
cancel        3
change        3
policy        3
order_info    2

Сохранено: data/eval_suite.json (версия v1, 16 задач)


## Итоги

- Задача корзины — полностью описанный сценарий: запрос + начальное состояние + политики + **ожидаемое итоговое состояние** + разметка (категория, персона, сложность) + reference.
- Корзина собрана **вручную** и расширена **генерацией БЯМ** с валидацией; есть **негативные/двунаправленные** и граничные случаи, отслеживается **баланс классов**.
- Корзина **версионирована** и сохранена — это переиспользуемый артефакт для всех дальнейших прогонов.

**Дальше:** Пример 3 — инфраструктура прогонов (harness), программные грейдеры, Iron User и метрики стабильности pass@k / pass^k.